In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import seaborn as sns
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from warnings import filterwarnings
filterwarnings("ignore")

In [ ]:
#Set professional colour theme
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("viridis")

viridis_colors=cm.viridis(np.linspace(0, 1, 5))
primary_color=viridis_colors[0]
secondary_color=viridis_colors[1]
accent_color=viridis_colors[2]
danger_color='#800000'
neutral_color=viridis_colors[4]
custom_palette=viridis_colors

In [ ]:
df=pd.read_csv('DataCoSupplyChainDataset.csv', encoding='latin-1')

In [ ]:
df.columns

In [ ]:
df['Benefit per order']==df['Order Profit Per Order']

## EDA

In [ ]:
print('rows, cols:', df.shape)
print('\ncolumns:')
print(df.columns.tolist())
print('\nNum Duplicates:', df.duplicated().sum())
print('\nMissing Values (top 20):')
print(df.isna().sum().sort_values(ascending=False).head(20))


In [ ]:
# Cleaning data
columns_to_drop = [
    'Customer Id', 'Customer Email', 'Product Description', 'Product Image',
    'Customer State', 'Order Customer Id', 'Customer City', 'Customer Fname',
    'Customer Lname', 'Customer Password', 'Customer Street', 'Customer Zipcode',
    'Latitude', 'Longitude', 'Order Item Cardprod Id', 'Order Item Discount',
    'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price',
    'Order Item Total', 'Order Item Quantity', 'Order Zipcode', 'Department Id',
    'Category Id', 'Order Id', 'Product Card Id', 'Product Category Id',
    'Product Status', 'Benefit per order', 'Order City', 'Order Country',
    'Order State', 'Market'
]

In [ ]:
#Dropping columns that are either fully missing, redundant, or have only one value
columns_to_drop_existing = [col for col in columns_to_drop if col in df.columns]

# See which ones were NOT found (for debugging)
missing = [col for col in columns_to_drop if col not in df.columns]
print("Columns NOT found in dataframe:", missing)

df = df.drop(columns=columns_to_drop_existing)

In [ ]:
#Removing cancelled order since they are not relevant for delivery time analysis and may have different patterns than completed orders
df=df[df['Delivery Status']!='Shipping canceled']

In [ ]:
#After changing data set, lets now have overview again
print('rows,cols:',df.shape)
print('\nMissing Values (top 5):')
print(df.isna().sum().sort_values(ascending=False).head())

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
df.head()

In [ ]:
#Value counts for categorical columns with low cardinality
for col in df.columns:
    if df[col].nunique()<10:
        print(f'\n{col},value counts:')
        print(df[col].value_counts())

In [ ]:
df.columns

In [ ]:
df['Order Profit Per Order']>0

In [ ]:
# Corrected code:
df['Profitability Flag'] = np.where(
    df['Order Profit Per Order'] > 0, 'Profit',
    np.where(df['Order Profit Per Order'] < 0, 'Loss', 'Break_even')
)

df['Profitability Flag'].value_counts()

In [ ]:
df['Profitability Flag'].value_counts(normalize=True)

In [ ]:
#Visualization of profitability distribution
profit_counts=df['Profitability Flag'].value_counts(normalize=True) * 100
profit_counts.plot(kind='pie', autopct='%1.1f%%' ,colors=[accent_color, danger_color, secondary_color])
plt.ylabel('')
plt.title('Profitability Distribution (%)')
plt.show()

In [ ]:
# Ensure date columns are datetime
df['order date (DateOrders)'] = pd.to_datetime(df['order date (DateOrders)'], errors='coerce')
df['shipping date (DateOrders)'] = pd.to_datetime(df['shipping date (DateOrders)'], errors='coerce')

# Calculate processing time and delay
df['Order Processing Time'] = (df['shipping date (DateOrders)'] - df['order date (DateOrders)']).dt.days
df['Delay'] = df['Order Processing Time'] - df['Days for shipment (scheduled)']
df['Is_Delayed'] = df['Delay'] > 0

In [ ]:
def format_func(value):
    if value >= 1e6:
        return f'{value/1e6:.1f}M $'
    elif value >= 1e3:
        return f'{value/1e3:.1f}K $'
    else:
        return f'{value:.0f} $'
delayed_df=df[df['Delay'] > 0]
metrics={}
metrics['Total Orders']=len(df)
metrics['Late Deliveries']=len(delayed_df)
metrics['90% Delay (days)']=delayed_df['Delay'].quantile(0.90)
metrics['On time Delivery %']=(1-float(metrics['Late Deliveries']) / metrics['Total Orders']) * 100
metrics['Late Delivery %']=float(metrics['Late Deliveries']) / metrics['Total Orders'] * 100
metrics['Total Profit']=format_func(df.loc[df['Order Profit Per Order'] > 0, 'Order Profit Per Order'].sum())
metrics['Total Loss due to delays']=format_func(df.loc[df['Delay'] > 0, 'Order Profit Per Order'].sum())
print('\n ---Business KPIs ---\n')
for k,v in metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.2f}")
    else:
        print(f"{k}: {v}")

### Profitability Vs. Delivery Time Analysis

In [ ]:
profit_metrics=(
    df.groupby('Delay')['Order Profit Per Order']
      .agg(
          mean_profit='mean',
          total_profit='sum',
          order_count='count'
      )
    .reset_index()
)

In [ ]:
profit_metrics

In [ ]:
delay_distribution=(
    df['Delay']
        .value_counts(normalize=True)
        .sort_index() * 100
).reset_index()

In [ ]:
delay_distribution

In [ ]:

# These values come directly from the notebook outputs
delay_distribution = pd.DataFrame({
    'Delay_Days': [-2, -1, 0, 1, 2, 3, 4],
    'Percentage': [12.08, 11.99, 21.21, 30.97, 15.95, 3.92, 3.88]
})

profit_metrics = pd.DataFrame({
    'Delay':        [-2,      -1,      0,       1,       2,       3,      4],
    'mean_profit':  [23.36,   21.60,   22.25,   22.33,   21.13,   20.03,  21.37],
    'total_profit': [487596,  447629,  815430,  1194895, 582111,  135653, 143107],
    'order_count':  [20873,   20719,   36650,   53503,   27551,   6772,   6697]
})

# ── Figure Setup ──────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Delay Distribution & Profit Analysis by Delay Days',
             fontsize=15, fontweight='bold', y=1.01)

# ── Subplot 1 : Delay Distribution ───────────────────────────────────────────
sns.barplot(
    x='Delay_Days', y='Percentage',
    data=delay_distribution,
    color=accent_color, ax=ax1
)
ax1.set_title('Delay Distribution', fontsize=13, fontweight='bold')
ax1.set_xlabel('Delay (days)', fontsize=11)
ax1.set_ylabel('Percentage of Orders (%)', fontsize=11)

# Annotate percentage on top of each bar
for bar in ax1.patches:
    height = bar.get_height()
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.5,
        f'{height:.1f}%',
        ha='center', va='bottom',
        fontsize=9, fontweight='bold'
    )

ax1.set_ylim(0, delay_distribution['Percentage'].max() * 1.18)

# ── Subplot 2 : Profit Analysis by Delay Days (twin-axis) ────────────────────
delays = profit_metrics['Delay'].astype(str)
x      = np.arange(len(delays))
width  = 0.35

# Primary y-axis — Total Profit bars
bars_profit = ax2.bar(
    x - width / 2,
    profit_metrics['total_profit'] / 1e6,   # convert to millions
    width,
    color=primary_color, label='Total Profit (M)'
)
ax2.set_ylabel('Total Profit ($M)', color=primary_color, fontsize=11)
ax2.tick_params(axis='y', labelcolor=primary_color)
ax2.set_title('Profit Analysis by Delay Days', fontsize=13, fontweight='bold')
ax2.set_xlabel('Delay (days)', fontsize=11)
ax2.set_xticks(x)
ax2.set_xticklabels(delays)

# Annotate total profit bars
for bar in bars_profit:
    height = bar.get_height()
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.01,
        f'${height:.2f}M',
        ha='center', va='bottom',
        fontsize=8, color=primary_color
    )

# Secondary y-axis — Mean Profit bars
ax2b = ax2.twinx()
bars_mean = ax2b.bar(
    x + width / 2,
    profit_metrics['mean_profit'],
    width,
    color=secondary_color, label='Mean Profit ($)'
)
ax2b.set_ylabel('Mean Profit Per Order ($)', color=secondary_color, fontsize=11)
ax2b.tick_params(axis='y', labelcolor=secondary_color)

# Annotate mean profit bars
for bar in bars_mean:
    height = bar.get_height()
    ax2b.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.2,
        f'${height:.1f}',
        ha='center', va='bottom',
        fontsize=8, color=secondary_color
    )

# Combined legend
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2b.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2,
           loc='upper right', fontsize=9)

# ── Final Layout ──────────────────────────────────────────────────────────────
plt.tight_layout()
plt.savefig('supply_chain_delay_viz.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved as supply_chain_delay_viz.png")

In [ ]:
df['Delay'] = df['Days for shipping (real)'] - df['Days for shipment (scheduled)']
df['Is_Late'] = (df['Delay'] > 0).astype(int)
shipping_bn = (
    df.groupby('Shipping Mode')['Is_Late']
    .agg(late_rate='mean', volume='count')
    .reset_index()
    .assign(late_rate=lambda x: x['late_rate'] * 100)
    .sort_values('late_rate', ascending=False)
)
 
#  Late-delivery rate by Order Region
region_bn = (
    df.groupby('Order Region')['Is_Late']
    .agg(late_rate='mean', volume='count')
    .reset_index()
    .assign(late_rate=lambda x: x['late_rate'] * 100)
    .sort_values('late_rate', ascending=False)
)
 
#  Late-delivery rate by Department
dept_bn = (
    df.groupby('Department Name')['Is_Late']
    .agg(late_rate='mean', volume='count')
    .reset_index()
    .assign(late_rate=lambda x: x['late_rate'] * 100)
    .sort_values('late_rate', ascending=False)
)
 
#  Late-delivery rate by Customer Segment
seg_bn = (
    df.groupby('Customer Segment')['Is_Late']
    .agg(late_rate='mean', volume='count')
    .reset_index()
    .assign(late_rate=lambda x: x['late_rate'] * 100)
    .sort_values('late_rate', ascending=False)
)
 
#  Mean delay days by Shipping Mode × Department (heatmap matrix)
heatmap_data = (
    df.groupby(['Shipping Mode', 'Department Name'])['Delay']
    .mean()
    .unstack('Department Name')
)
 

#  BOTTLENECK THRESHOLD  — flag anything above overall late rate
overall_late_rate = df['Is_Late'].mean() * 100
print(f"\n{'='*55}")
print(f"  Overall Late-Delivery Rate : {overall_late_rate:.1f}%")
print(f"{'='*55}")
 
def flag(rate, threshold=overall_late_rate):
    return "BOTTLENECK" if rate > threshold else "OK"
 
print("\n── Shipping Mode Bottlenecks ──")
for _, row in shipping_bn.iterrows():
    print(f"  {row['Shipping Mode']:<20} {row['late_rate']:5.1f}%  {flag(row['late_rate'])}")
 
print("\n── Top 5 Region Bottlenecks ──")
for _, row in region_bn.head(5).iterrows():
    print(f"  {row['Order Region']:<25} {row['late_rate']:5.1f}%  {flag(row['late_rate'])}")
 
print("\n── Department Bottlenecks ──")
for _, row in dept_bn.iterrows():
    print(f"  {row['Department Name']:<25} {row['late_rate']:5.1f}%  {flag(row['late_rate'])}")
 
print("\n── Customer Segment Bottlenecks ──")
for _, row in seg_bn.iterrows():
    print(f"  {row['Customer Segment']:<25} {row['late_rate']:5.1f}%  {flag(row['late_rate'])}")
 

#  VISUALISATIONS  — 3 × 2 dashboard

fig = plt.figure(figsize=(20, 16))
fig.suptitle('Supply Chain Bottleneck Detection', fontsize=18,
             fontweight='bold', y=0.98)
 
gs = fig.add_gridspec(3, 2, hspace=0.55, wspace=0.35)
 
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, 0])
ax4 = fig.add_subplot(gs[1, 1])
ax5 = fig.add_subplot(gs[2, :])   # full-width heatmap
 
# Helper : colour bars red if above threshold 
def bar_colors(values, threshold):
    return [danger_color if v > threshold else accent_color for v in values]
 
# 1 · Shipping Mode 
colors1 = bar_colors(shipping_bn['late_rate'], overall_late_rate)
bars1 = ax1.barh(shipping_bn['Shipping Mode'], shipping_bn['late_rate'],
                 color=colors1, edgecolor='white', linewidth=0.6)
ax1.axvline(overall_late_rate, color='black', linestyle='--', linewidth=1.2,
            label=f'Avg {overall_late_rate:.1f}%')
ax1.set_title('Late-Delivery Rate by Shipping Mode', fontweight='bold', fontsize=11)
ax1.set_xlabel('Late-Delivery Rate (%)')
for bar in bars1:
    w = bar.get_width()
    ax1.text(w + 0.5, bar.get_y() + bar.get_height() / 2,
             f'{w:.1f}%', va='center', fontsize=9)
ax1.legend(fontsize=8)
 
# 2 · Customer Segment
colors2 = bar_colors(seg_bn['late_rate'], overall_late_rate)
bars2 = ax2.barh(seg_bn['Customer Segment'], seg_bn['late_rate'],
                 color=colors2, edgecolor='white', linewidth=0.6)
ax2.axvline(overall_late_rate, color='black', linestyle='--', linewidth=1.2,
            label=f'Avg {overall_late_rate:.1f}%')
ax2.set_title('Late-Delivery Rate by Customer Segment', fontweight='bold', fontsize=11)
ax2.set_xlabel('Late-Delivery Rate (%)')
for bar in bars2:
    w = bar.get_width()
    ax2.text(w + 0.5, bar.get_y() + bar.get_height() / 2,
             f'{w:.1f}%', va='center', fontsize=9)
ax2.legend(fontsize=8)
 
# 3 · Department 
colors3 = bar_colors(dept_bn['late_rate'], overall_late_rate)
bars3 = ax3.barh(dept_bn['Department Name'], dept_bn['late_rate'],
                 color=colors3, edgecolor='white', linewidth=0.6)
ax3.axvline(overall_late_rate, color='black', linestyle='--', linewidth=1.2,
            label=f'Avg {overall_late_rate:.1f}%')
ax3.set_title('Late-Delivery Rate by Department', fontweight='bold', fontsize=11)
ax3.set_xlabel('Late-Delivery Rate (%)')
for bar in bars3:
    w = bar.get_width()
    ax3.text(w + 0.5, bar.get_y() + bar.get_height() / 2,
             f'{w:.1f}%', va='center', fontsize=9)
ax3.legend(fontsize=8)
 
# 4 · Top 10 Regions 
top10_regions = region_bn.head(10)
colors4 = bar_colors(top10_regions['late_rate'], overall_late_rate)
bars4 = ax4.barh(top10_regions['Order Region'][::-1],
                 top10_regions['late_rate'][::-1],
                 color=colors4[::-1], edgecolor='white', linewidth=0.6)
ax4.axvline(overall_late_rate, color='black', linestyle='--', linewidth=1.2,
            label=f'Avg {overall_late_rate:.1f}%')
ax4.set_title('Late-Delivery Rate – Top 10 Regions', fontweight='bold', fontsize=11)
ax4.set_xlabel('Late-Delivery Rate (%)')
for bar in bars4:
    w = bar.get_width()
    ax4.text(w + 0.5, bar.get_y() + bar.get_height() / 2,
             f'{w:.1f}%', va='center', fontsize=9)
ax4.legend(fontsize=8)
 
# 5 · Heatmap: Mean Delay (Shipping Mode × Department) 
sns.heatmap(
    heatmap_data,
    ax=ax5,
    cmap='RdYlGn_r',      # red = more delay, green = early/on-time
    annot=True,
    fmt='.2f',
    linewidths=0.4,
    linecolor='white',
    cbar_kws={'label': 'Mean Delay (days)', 'shrink': 0.6}
)
ax5.set_title('Mean Delay Days — Shipping Mode × Department  (red = bottleneck)',
              fontweight='bold', fontsize=11)
ax5.set_xlabel('Department Name')
ax5.set_ylabel('Shipping Mode')
ax5.tick_params(axis='x', rotation=30)
ax5.tick_params(axis='y', rotation=0)
 
Legend patch
red_patch   = mpatches.Patch(color=danger_color,  label='Bottleneck (above avg)')
green_patch = mpatches.Patch(color=accent_color,  label='Within average')
fig.legend(handles=[red_patch, green_patch],
           loc='lower center', ncol=2, fontsize=10,
           bbox_to_anchor=(0.5, -0.01), frameon=True)
 
plt.savefig('bottleneck_detection.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nPlot saved as bottleneck_detection.png")
 

In [ ]:
# ── Date features (run this before the RCA blocks) ───────────────────────────
df['order date (DateOrders)'] = pd.to_datetime(df['order date (DateOrders)'], errors='coerce')

df['order_month'] = df['order date (DateOrders)'].dt.month
df['order_day']   = df['order date (DateOrders)'].dt.day_name()
df['order_hour']  = df['order date (DateOrders)'].dt.hour
# RCA 1 · Shipping Mode — late rate + mean delay
rca_shipping = (
    df.groupby('Shipping Mode')
    .agg(late_rate=('Is_Late', 'mean'),
         mean_delay=('Delay', 'mean'),
         volume=('Is_Late', 'count'))
    .assign(late_rate=lambda x: x['late_rate'] * 100)
    .sort_values('late_rate', ascending=False)
    .reset_index()
)
 
# RCA 2 · Order Status — % of late orders per status
rca_status = (
    df.groupby('Order Status')
    .agg(late_rate=('Is_Late', 'mean'),
         mean_delay=('Delay', 'mean'),
         volume=('Is_Late', 'count'))
    .assign(late_rate=lambda x: x['late_rate'] * 100)
    .sort_values('late_rate', ascending=False)
    .reset_index()
)
 
# RCA 3 · Day of week — when are orders placed that become late?
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
rca_day = (
    df.groupby('order_day')['Is_Late']
    .mean()
    .reindex(day_order)
    .reset_index()
    .rename(columns={'Is_Late': 'late_rate'})
    .assign(late_rate=lambda x: x['late_rate'] * 100)
)
 
# RCA 4 · Month — seasonality of delays
rca_month = (
    df.groupby('order_month')['Is_Late']
    .mean()
    .reset_index()
    .rename(columns={'Is_Late': 'late_rate'})
    .assign(late_rate=lambda x: x['late_rate'] * 100)
)
month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
rca_month['month_name'] = rca_month['order_month'].map(month_names)
 
# RCA 5 · Payment Type vs Late rate
rca_payment = (
    df.groupby('Type')['Is_Late']
    .mean()
    .reset_index()
    .rename(columns={'Is_Late': 'late_rate'})
    .assign(late_rate=lambda x: x['late_rate'] * 100)
    .sort_values('late_rate', ascending=False)
)
 
# RCA 6 · Top 10 worst products (by mean delay, min 200 orders)
rca_product = (
    df.groupby('Product Name')
    .agg(mean_delay=('Delay', 'mean'), volume=('Delay', 'count'))
    .query('volume >= 200')
    .sort_values('mean_delay', ascending=False)
    .head(10)
    .reset_index()
)
 
# RCA 7 · Scheduled vs Real shipping days gap (schedule accuracy)
rca_schedule = (
    df.groupby('Days for shipment (scheduled)')
    .agg(mean_real=('Days for shipping (real)', 'mean'),
         late_rate=('Is_Late', 'mean'),
         volume=('Is_Late', 'count'))
    .assign(late_rate=lambda x: x['late_rate'] * 100)
    .reset_index()
)
 

#  CONSOLE SUMMARY

print(f"\n{'═'*60}")
print(f"  ROOT CAUSE ANALYSIS — Supply Chain Late Deliveries")
print(f"  Overall Late Rate : {overall_late_rate:.1f}%")
print(f"{'═'*60}")
 
print("\n── RCA 1 · Shipping Mode ──")
for _, r in rca_shipping.iterrows():
    flag = "[HIGH]" if r['late_rate'] > overall_late_rate else "[LOW]"
    print(f"  {flag} {r['Shipping Mode']:<20} late={r['late_rate']:.1f}%  mean_delay={r['mean_delay']:.2f}d  vol={r['volume']:,}")

print("\n── RCA 2 · Order Status ──")
for _, r in rca_status.iterrows():
    flag = "[HIGH]" if r['late_rate'] > overall_late_rate else "[LOW]"
    print(f"  {flag} {r['Order Status']:<20} late={r['late_rate']:.1f}%  mean_delay={r['mean_delay']:.2f}d")

print("\n── RCA 3 · Day of Week ──")
for _, r in rca_day.iterrows():
    flag = "[HIGH]" if r['late_rate'] > overall_late_rate else "[LOW]"
    print(f"  {flag} {r['order_day']:<12} late={r['late_rate']:.1f}%")

print("\n── RCA 5 · Payment Type ──")
for _, r in rca_payment.iterrows():
    flag = "[HIGH]" if r['late_rate'] > overall_late_rate else "[LOW]"
    print(f"  {flag} {r['Type']:<12} late={r['late_rate']:.1f}%")

print("\n── RCA 6 · Top 10 Products by Mean Delay ──")
for _, r in rca_product.iterrows():
    print(f"  [HIGH] {r['Product Name'][:40]:<40}  mean_delay={r['mean_delay']:.2f}d  vol={r['volume']:,}")
#  DASHBOARD  — 4 × 2 grid

fig = plt.figure(figsize=(22, 20))
fig.suptitle('Root Cause Analysis — Supply Chain Late Deliveries',
             fontsize=18, fontweight='bold', y=0.99)
 
gs = fig.add_gridspec(4, 2, hspace=0.60, wspace=0.38)
 
ax1 = fig.add_subplot(gs[0, 0])   # Shipping Mode
ax2 = fig.add_subplot(gs[0, 1])   # Order Status
ax3 = fig.add_subplot(gs[1, 0])   # Day of Week
ax4 = fig.add_subplot(gs[1, 1])   # Monthly seasonality
ax5 = fig.add_subplot(gs[2, 0])   # Payment type
ax6 = fig.add_subplot(gs[2, 1])   # Top products
ax7 = fig.add_subplot(gs[3, :])   # Schedule accuracy (full width)
 
def bar_colors(values, threshold=overall_late_rate):
    return [danger_color if v > threshold else accent_color for v in values]
 
# 1 · Shipping Mode 
c1 = bar_colors(rca_shipping['late_rate'])
b1 = ax1.bar(rca_shipping['Shipping Mode'], rca_shipping['late_rate'],
             color=c1, edgecolor='white')
ax1.axhline(overall_late_rate, color='black', linestyle='--', linewidth=1.2,
            label=f'Avg {overall_late_rate:.1f}%')
ax1.set_title('Late Rate by Shipping Mode', fontweight='bold')
ax1.set_ylabel('Late-Delivery Rate (%)')
ax1.set_xlabel('')
for bar in b1:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9)
ax1.legend(fontsize=8)
ax1.tick_params(axis='x', rotation=15)
 
# 2 · Order Status 
c2 = bar_colors(rca_status['late_rate'])
b2 = ax2.barh(rca_status['Order Status'][::-1], rca_status['late_rate'][::-1],
              color=c2[::-1], edgecolor='white')
ax2.axvline(overall_late_rate, color='black', linestyle='--', linewidth=1.2,
            label=f'Avg {overall_late_rate:.1f}%')
ax2.set_title('Late Rate by Order Status', fontweight='bold')
ax2.set_xlabel('Late-Delivery Rate (%)')
for bar in b2:
    w = bar.get_width()
    ax2.text(w + 0.5, bar.get_y() + bar.get_height()/2,
             f'{w:.1f}%', va='center', fontsize=9)
ax2.legend(fontsize=8)
 
# 3 · Day of Week 
c3 = bar_colors(rca_day['late_rate'])
b3 = ax3.bar(rca_day['order_day'], rca_day['late_rate'],
             color=c3, edgecolor='white')
ax3.axhline(overall_late_rate, color='black', linestyle='--', linewidth=1.2,
            label=f'Avg {overall_late_rate:.1f}%')
ax3.set_title('Late Rate by Day Orders Were Placed', fontweight='bold')
ax3.set_ylabel('Late-Delivery Rate (%)')
for bar in b3:
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8)
ax3.legend(fontsize=8)
ax3.tick_params(axis='x', rotation=30)
 
# 4 · Monthly Seasonality 
c4 = bar_colors(rca_month['late_rate'])
b4 = ax4.bar(rca_month['month_name'], rca_month['late_rate'],
             color=c4, edgecolor='white')
ax4.axhline(overall_late_rate, color='black', linestyle='--', linewidth=1.2,
            label=f'Avg {overall_late_rate:.1f}%')
ax4.set_title('Late Rate by Month (Seasonality)', fontweight='bold')
ax4.set_ylabel('Late-Delivery Rate (%)')
for bar in b4:
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8)
ax4.legend(fontsize=8)
 
# 5 · Payment Type 
c5 = bar_colors(rca_payment['late_rate'])
b5 = ax5.bar(rca_payment['Type'], rca_payment['late_rate'],
             color=c5, edgecolor='white')
ax5.axhline(overall_late_rate, color='black', linestyle='--', linewidth=1.2,
            label=f'Avg {overall_late_rate:.1f}%')
ax5.set_title('Late Rate by Payment Type', fontweight='bold')
ax5.set_ylabel('Late-Delivery Rate (%)')
for bar in b5:
    ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9)
ax5.legend(fontsize=8)
 
# 6 · Top 10 Worst Products 
short_names = [n[:28] + '…' if len(n) > 28 else n for n in rca_product['Product Name']]
b6 = ax6.barh(short_names[::-1], rca_product['mean_delay'][::-1],
              color=danger_color, edgecolor='white', alpha=0.85)
ax6.axvline(0, color='black', linewidth=0.8)
ax6.set_title('Top 10 Products by Mean Delay (days)', fontweight='bold')
ax6.set_xlabel('Mean Delay (days)')
for bar in b6:
    w = bar.get_width()
    ax6.text(w + 0.03, bar.get_y() + bar.get_height()/2,
             f'{w:.2f}d', va='center', fontsize=8)
 
# 7 · Schedule Accuracy (full width) 
x7    = rca_schedule['Days for shipment (scheduled)']
width = 0.35
b7a   = ax7.bar(x7 - width/2, rca_schedule['mean_real'], width,
                color=primary_color, label='Avg Real Shipping Days', alpha=0.9)
b7b   = ax7.bar(x7 + width/2, rca_schedule['late_rate'],  width,
                color=danger_color,  label='Late Rate (%)', alpha=0.9)
 
# Reference line: perfect schedule accuracy
ax7.plot(x7, x7, 'k--', linewidth=1.2, label='Perfect Schedule (real = scheduled)')
ax7.set_title('Schedule Accuracy — Scheduled vs Real Shipping Days & Late Rate',
              fontweight='bold')
ax7.set_xlabel('Scheduled Shipping Days')
ax7.set_ylabel('Days / Late Rate (%)')
ax7.set_xticks(x7)
for bar in b7a:
    ax7.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9,
             color=primary_color)
for bar in b7b:
    ax7.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9,
             color=danger_color)
ax7.legend(fontsize=9)
 
# Global legend 
red_patch   = mpatches.Patch(color=danger_color, label='Root Cause / Above Average')
green_patch = mpatches.Patch(color=accent_color, label='Within Average')
fig.legend(handles=[red_patch, green_patch],
           loc='lower center', ncol=2, fontsize=10,
           bbox_to_anchor=(0.5, -0.005), frameon=True)
 
plt.savefig('root_cause_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nPlot saved as root_cause_analysis.png")
 

In [ ]:

df['order date (DateOrders)']    = pd.to_datetime(df['order date (DateOrders)'],    errors='coerce')
df['shipping date (DateOrders)'] = pd.to_datetime(df['shipping date (DateOrders)'], errors='coerce')

df['Order Processing Time'] = (df['shipping date (DateOrders)'] - df['order date (DateOrders)']).dt.days
df['Delay']         = df['Order Processing Time'] - df['Days for shipment (scheduled)']
df['Is_Delayed']    = (df['Delay'] > 0).astype(int)

df['order_year']       = df['order date (DateOrders)'].dt.year
df['order_month']      = df['order date (DateOrders)'].dt.month
df['order_month_name'] = df['order date (DateOrders)'].dt.strftime('%b')
df['order_quarter']    = df['order date (DateOrders)'].dt.quarter
df['order_day']        = df['order date (DateOrders)'].dt.day_name()
df['order_hour']       = df['order date (DateOrders)'].dt.hour
df['YearMonth']        = df['order date (DateOrders)'].dt.to_period('M')

print("Columns added:", ['Order Processing Time','Delay','Is_Delayed',
                            'order_year','order_month','order_month_name',
                            'order_quarter','order_day','order_hour','YearMonth'])

In [ ]:
# 1 · Monthly trend — orders, late rate, avg processing time
monthly = (
    df.groupby('YearMonth')
    .agg(
        order_count   = ('Is_Delayed', 'count'),
        late_rate     = ('Is_Delayed', 'mean'),
        avg_proc_time = ('Order Processing Time', 'mean'),
        total_profit  = ('Order Profit Per Order', 'sum')
    )
    .reset_index()
)
monthly['late_rate']    *= 100
monthly['YearMonth_dt']  = monthly['YearMonth'].dt.to_timestamp()
 
# 2 · Day-of-week pattern
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow = (
    df.groupby('order_day')
    .agg(order_count=('Is_Delayed','count'), late_rate=('Is_Delayed','mean'))
    .reindex(day_order)
    .reset_index()
)
dow['late_rate'] *= 100
 
# 3 · Hour-of-day pattern
hourly = (
    df.groupby('order_hour')
    .agg(order_count=('Is_Delayed','count'), late_rate=('Is_Delayed','mean'))
    .reset_index()
)
hourly['late_rate'] *= 100
 
# 4 · Monthly seasonality (avg across years)
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
seasonal = (
    df.groupby('order_month_name')
    .agg(late_rate=('Is_Delayed','mean'), avg_delay=('Delay','mean'))
    .reindex(month_order)
    .reset_index()
)
seasonal['late_rate'] *= 100
 
# 5 · Quarterly profit & late rate
quarterly = (
    df.groupby(['order_year','order_quarter'])
    .agg(
        total_profit = ('Order Profit Per Order', 'sum'),
        late_rate    = ('Is_Delayed', 'mean')
    )
    .reset_index()
)
quarterly['late_rate']   *= 100
quarterly['Quarter_label'] = quarterly['order_year'].astype(str) + ' Q' + quarterly['order_quarter'].astype(str)
 

#  PRINT SUMMARY

print(f"\n{'='*55}")
print("  TIME-BASED ANALYSIS SUMMARY")
print(f"{'='*55}")
print(f"  Date range   : {df['order date (DateOrders)'].min().date()}  →  {df['order date (DateOrders)'].max().date()}")
print(f"  Total orders : {len(df):,}")
print(f"  Avg late rate: {df['Is_Delayed'].mean()*100:.1f}%")
print(f"\n  Busiest month (orders)   : {monthly.loc[monthly['order_count'].idxmax(), 'YearMonth']}")
print(f"  Worst month (late rate)  : {monthly.loc[monthly['late_rate'].idxmax(), 'YearMonth']}  "
      f"({monthly['late_rate'].max():.1f}%)")
print(f"  Best month  (late rate)  : {monthly.loc[monthly['late_rate'].idxmin(), 'YearMonth']}  "
      f"({monthly['late_rate'].min():.1f}%)")
print(f"\n  Worst day-of-week        : {dow.loc[dow['late_rate'].idxmax(), 'order_day']}  "
      f"({dow['late_rate'].max():.1f}%)")
print(f"  Peak order hour          : {hourly.loc[hourly['order_count'].idxmax(), 'order_hour']}:00")
print(f"{'='*55}\n")
 

#  VISUALISATIONS  — 3 × 2 dashboard

fig = plt.figure(figsize=(20, 18))
fig.suptitle('Supply Chain — Time-Based Analysis', fontsize=18, fontweight='bold', y=0.98)
 
gs = fig.add_gridspec(3, 2, hspace=0.55, wspace=0.35)
ax1 = fig.add_subplot(gs[0, :])   # full-width monthly trend
ax2 = fig.add_subplot(gs[1, 0])
ax3 = fig.add_subplot(gs[1, 1])
ax4 = fig.add_subplot(gs[2, 0])
ax5 = fig.add_subplot(gs[2, 1])
 
# 1 · Monthly Order Volume + Late Rate
ax1b = ax1.twinx()
 
ax1.bar(monthly['YearMonth_dt'], monthly['order_count'],
        color=accent_color, alpha=0.6, label='Order Volume', width=20)
ax1b.plot(monthly['YearMonth_dt'], monthly['late_rate'],
          color=danger_color, linewidth=2, marker='o', markersize=4, label='Late Rate %')
 
ax1.set_title('Monthly Order Volume & Late-Delivery Rate Trend', fontweight='bold', fontsize=12)
ax1.set_xlabel('Month')
ax1.set_ylabel('Order Volume', color=accent_color, fontsize=10)
ax1b.set_ylabel('Late-Delivery Rate (%)', color=danger_color, fontsize=10)
ax1.tick_params(axis='y', labelcolor=accent_color)
ax1b.tick_params(axis='y', labelcolor=danger_color)
 
# Combined legend
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax1b.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc='upper left', fontsize=9)
 
# 2 · Day-of-Week Late Rate 
overall_late = df['Is_Delayed'].mean() * 100
colors_dow = [danger_color if v > overall_late else accent_color for v in dow['late_rate']]
 
bars2 = ax2.bar(dow['order_day'], dow['late_rate'], color=colors_dow,
                edgecolor='white', linewidth=0.6)
ax2.axhline(overall_late, color='black', linestyle='--', linewidth=1.2,
            label=f'Avg {overall_late:.1f}%')
ax2.set_title('Late-Delivery Rate by Day of Week', fontweight='bold', fontsize=11)
ax2.set_xlabel('Day of Week')
ax2.set_ylabel('Late-Delivery Rate (%)')
ax2.tick_params(axis='x', rotation=30)
for bar in bars2:
    h = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, h + 0.2,
             f'{h:.1f}%', ha='center', va='bottom', fontsize=8)
ax2.legend(fontsize=8)
 
# 3 · Hourly Order Volume 
ax3b = ax3.twinx()
 
ax3.bar(hourly['order_hour'], hourly['order_count'],
        color=primary_color, alpha=0.7, label='Order Volume')
ax3b.plot(hourly['order_hour'], hourly['late_rate'],
          color=danger_color, linewidth=2, marker='o', markersize=4, label='Late Rate %')
 
ax3.set_title('Order Volume & Late Rate by Hour of Day', fontweight='bold', fontsize=11)
ax3.set_xlabel('Hour of Day')
ax3.set_ylabel('Order Volume', color=primary_color, fontsize=10)
ax3b.set_ylabel('Late-Delivery Rate (%)', color=danger_color, fontsize=10)
ax3.tick_params(axis='y', labelcolor=primary_color)
ax3b.tick_params(axis='y', labelcolor=danger_color)
ax3.set_xticks(range(0, 24, 2))
h1, l1 = ax3.get_legend_handles_labels()
h2, l2 = ax3b.get_legend_handles_labels()
ax3.legend(h1 + h2, l1 + l2, loc='upper left', fontsize=8)
 
# 4 · Monthly Seasonality (avg late rate per calendar month) 
colors_sea = [danger_color if v > overall_late else accent_color
              for v in seasonal['late_rate'].fillna(0)]
bars4 = ax4.bar(seasonal['order_month_name'], seasonal['late_rate'],
                color=colors_sea, edgecolor='white', linewidth=0.6)
ax4.axhline(overall_late, color='black', linestyle='--', linewidth=1.2,
            label=f'Avg {overall_late:.1f}%')
ax4.set_title('Seasonal Late-Delivery Rate (by Calendar Month)', fontweight='bold', fontsize=11)
ax4.set_xlabel('Month')
ax4.set_ylabel('Late-Delivery Rate (%)')
for bar in bars4:
    h = bar.get_height()
    if h > 0:
        ax4.text(bar.get_x() + bar.get_width()/2, h + 0.2,
                 f'{h:.1f}%', ha='center', va='bottom', fontsize=8)
ax4.legend(fontsize=8)
 
# 5 · Quarterly Profit & Late Rate
ax5b = ax5.twinx()
 
ax5.bar(range(len(quarterly)), quarterly['total_profit'] / 1e6,
        color=secondary_color, alpha=0.7, label='Total Profit ($M)')
ax5b.plot(range(len(quarterly)), quarterly['late_rate'],
          color=danger_color, linewidth=2, marker='s', markersize=5, label='Late Rate %')
 
ax5.set_title('Quarterly Profit vs Late-Delivery Rate', fontweight='bold', fontsize=11)
ax5.set_xlabel('Quarter')
ax5.set_ylabel('Total Profit ($M)', color=secondary_color, fontsize=10)
ax5b.set_ylabel('Late-Delivery Rate (%)', color=danger_color, fontsize=10)
ax5.tick_params(axis='y', labelcolor=secondary_color)
ax5b.tick_params(axis='y', labelcolor=danger_color)
ax5.set_xticks(range(len(quarterly)))
ax5.set_xticklabels(quarterly['Quarter_label'], rotation=45, ha='right', fontsize=7)
h1, l1 = ax5.get_legend_handles_labels()
h2, l2 = ax5b.get_legend_handles_labels()
ax5.legend(h1 + h2, l1 + l2, loc='upper left', fontsize=8)
 
# Legend patch
red_patch   = mpatches.Patch(color=danger_color, label='Above average late rate')
green_patch = mpatches.Patch(color=accent_color, label='Within average late rate')
fig.legend(handles=[red_patch, green_patch],
           loc='lower center', ncol=2, fontsize=10,
           bbox_to_anchor=(0.5, -0.01), frameon=True)
 
plt.savefig('time_based_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved as time_based_analysis.png")
 

In [ ]:

df['order date (DateOrders)']    = pd.to_datetime(df['order date (DateOrders)'],    errors='coerce')
df['shipping date (DateOrders)'] = pd.to_datetime(df['shipping date (DateOrders)'], errors='coerce')

df['Order Processing Time'] = (df['shipping date (DateOrders)'] - df['order date (DateOrders)']).dt.days
df['Delay']      = df['Order Processing Time'] - df['Days for shipment (scheduled)']
df['Is_Delayed'] = (df['Delay'] > 0).astype(int)

df['order_month']   = df['order date (DateOrders)'].dt.month
df['order_quarter'] = df['order date (DateOrders)'].dt.quarter
df['order_dow']     = df['order date (DateOrders)'].dt.dayofweek   # ← was missing
df['order_hour']    = df['order date (DateOrders)'].dt.hour

print(" All features ready. Columns added:", 
      ['Order Processing Time','Delay','Is_Delayed',
       'order_month','order_quarter','order_dow','order_hour'])

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, ConfusionMatrixDisplay
)
#  FEATURE SELECTION
#  Only use columns available AT ORDER TIME (no leakage from shipping/delivery)
FEATURES = [
    # Order-level numeric
    'Days for shipment (scheduled)',
    'Sales per customer',
    'Sales',
    'Order Item Profit Ratio',
    'Order Profit Per Order',
    'Product Price',
    # Time features
    'order_month',
    'order_quarter',
    'order_dow',
    'order_hour',
    # Categorical (will be encoded)
    'Type',                  # payment type
    'Customer Segment',
    'Customer Country',
    'Department Name',
    'Category Name',
    'Order Region',
    'Order Status',
    'Shipping Mode',
]
 
TARGET = 'Is_Delayed'
 
model_df = df[FEATURES + [TARGET]].dropna()
 
print(f"\n{'='*55}")
print("  PREDICTIVE MODEL: Late Delivery Classification")
print(f"{'='*55}")
print(f"  Total samples : {len(model_df):,}")
print(f"  Late (1)      : {model_df[TARGET].sum():,}  ({model_df[TARGET].mean()*100:.1f}%)")
print(f"  On-Time (0)   : {(model_df[TARGET]==0).sum():,}  ({(model_df[TARGET]==0).mean()*100:.1f}%)")
print(f"  Features used : {len(FEATURES)}")
print(f"{'='*55}\n")
 
#  ENCODING
CAT_COLS = ['Type', 'Customer Segment', 'Customer Country',
            'Department Name', 'Category Name', 'Order Region',
            'Order Status', 'Shipping Mode']
 
le = LabelEncoder()
for col in CAT_COLS:
    model_df[col] = le.fit_transform(model_df[col].astype(str))
 

#  TRAIN / TEST SPLIT

X = model_df[FEATURES]
y = model_df[TARGET]
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
 
# Scale for Logistic Regression
scaler  = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
 
#  MODELS
models = {
    'Logistic Regression' : LogisticRegression(max_iter=500, random_state=42),
    'Random Forest'        : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting'    : GradientBoostingClassifier(n_estimators=100, random_state=42),
}
 
results  = {}
cv       = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
 
for name, model in models.items():
    Xtr = X_train_sc if name == 'Logistic Regression' else X_train
    Xte = X_test_sc  if name == 'Logistic Regression' else X_test
 
    # Cross-validation
    cv_scores = cross_val_score(model, Xtr, y_train, cv=cv, scoring='roc_auc')
 
    # Fit & predict
    model.fit(Xtr, y_train)
    y_pred      = model.predict(Xte)
    y_prob      = model.predict_proba(Xte)[:, 1]
    roc_auc     = roc_auc_score(y_test, y_prob)
    report      = classification_report(y_test, y_pred, output_dict=True)
 
    results[name] = {
        'model'     : model,
        'y_pred'    : y_pred,
        'y_prob'    : y_prob,
        'roc_auc'   : roc_auc,
        'cv_mean'   : cv_scores.mean(),
        'cv_std'    : cv_scores.std(),
        'report'    : report,
        'X_test'    : Xte,
    }
 
    print(f"  {name}")
    print(f"    CV AUC  : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print(f"    Test AUC: {roc_auc:.4f}")
    print(f"    Accuracy: {report['accuracy']:.4f}")
    print(f"    F1 (Late): {report['1']['f1-score']:.4f}\n")
 

#  BEST MODEL

best_name  = max(results, key=lambda k: results[k]['roc_auc'])
best       = results[best_name]
print(f"   Best Model : {best_name}  (AUC = {best['roc_auc']:.4f})")
print(f"\n{classification_report(y_test, best['y_pred'], target_names=['On-Time','Late'])}")
 

#  FEATURE IMPORTANCE  (from Random Forest)

rf_model       = results['Random Forest']['model']
feat_imp       = pd.Series(rf_model.feature_importances_, index=FEATURES) \
                   .sort_values(ascending=False)
 
#  VISUALISATIONS  — 2 × 2 dashboard
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle('Late Delivery Prediction — Model Evaluation', fontsize=17,
             fontweight='bold', y=0.98)
 
# 1 · ROC Curves (all models) 
ax1   = axes[0, 0]
colors_roc = [primary_color, accent_color, secondary_color]
for (name, res), col in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    ax1.plot(fpr, tpr, color=col, linewidth=2,
             label=f"{name}  (AUC={res['roc_auc']:.3f})")
ax1.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Baseline')
ax1.set_title('ROC Curves — All Models', fontweight='bold', fontsize=12)
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.legend(fontsize=9)
ax1.set_xlim([0, 1])
ax1.set_ylim([0, 1.02])
 
# 2 · Confusion Matrix (best model) 
ax2 = axes[0, 1]
cm_matrix = confusion_matrix(y_test, best['y_pred'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm_matrix,
                               display_labels=['On-Time', 'Late'])
disp.plot(ax=ax2, colorbar=False, cmap='viridis')
ax2.set_title(f'Confusion Matrix — {best_name}', fontweight='bold', fontsize=12)
 
# 3 · Feature Importance (Random Forest top 12) 
ax3    = axes[1, 0]
top12  = feat_imp.head(12)
colors_fi = [danger_color if i < 3 else accent_color for i in range(len(top12))]
bars   = ax3.barh(top12.index[::-1], top12.values[::-1],
                  color=colors_fi[::-1], edgecolor='white')
ax3.set_title('Feature Importance — Random Forest (Top 12)',
              fontweight='bold', fontsize=12)
ax3.set_xlabel('Importance Score')
for bar in bars:
    w = bar.get_width()
    ax3.text(w + 0.001, bar.get_y() + bar.get_height() / 2,
             f'{w:.3f}', va='center', fontsize=8)
 
# 4 · Model Comparison Bar Chart (AUC + Accuracy) 
ax4       = axes[1, 1]
model_names = list(results.keys())
aucs        = [results[n]['roc_auc'] for n in model_names]
accs        = [results[n]['report']['accuracy'] for n in model_names]
x           = np.arange(len(model_names))
width       = 0.35
 
bars_auc = ax4.bar(x - width/2, aucs, width, color=primary_color,
                   label='ROC-AUC', edgecolor='white')
bars_acc = ax4.bar(x + width/2, accs, width, color=accent_color,
                   label='Accuracy', edgecolor='white')
 
ax4.set_title('Model Comparison — AUC vs Accuracy', fontweight='bold', fontsize=12)
ax4.set_ylabel('Score')
ax4.set_xticks(x)
ax4.set_xticklabels(model_names, fontsize=10)
ax4.set_ylim(0.5, 1.05)
ax4.legend(fontsize=9)
ax4.axhline(0.5, color='grey', linestyle='--', linewidth=0.8)
 
for bar in list(bars_auc) + list(bars_acc):
    h = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2, h + 0.005,
             f'{h:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
 
# Mark best model
best_idx = model_names.index(best_name)
ax4.annotate('★ Best', xy=(best_idx - width/2, aucs[best_idx]),
             xytext=(best_idx - width/2 + 0.05, aucs[best_idx] + 0.03),
             fontsize=9, color=danger_color, fontweight='bold')
 
plt.tight_layout()
plt.savefig('delivery_prediction_model.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nPlot saved as delivery_prediction_model.png")
 
#  PREDICT ON NEW ORDER  (example usage)

print("\n" + "="*55)
print("  EXAMPLE: Predict a Single New Order")
print("="*55)
 
new_order = pd.DataFrame([{
    'Days for shipment (scheduled)' : 4,
    'Sales per customer'            : 250.0,
    'Sales'                         : 199.99,
    'Order Item Profit Ratio'       : 0.15,
    'Order Profit Per Order'        : 30.0,
    'Product Price'                 : 199.99,
    'order_month'                   : 3,
    'order_quarter'                 : 1,
    'order_dow'                     : 0,        # Monday
    'order_hour'                    : 14,
    'Type'                          : 2,        # encoded integer
    'Customer Segment'              : 0,
    'Customer Country'              : 0,
    'Department Name'               : 2,
    'Category Name'                 : 5,
    'Order Region'                  : 3,
    'Order Status'                  : 1,
    'Shipping Mode'                 : 3,        # Standard Class
}])
 
best_model = best['model']
prob_late  = best_model.predict_proba(new_order)[0][1]
prediction = "LATE" if prob_late > 0.5 else "ON-TIME"
 
print(f"  Predicted      : {prediction}")
print(f"  Probability of Late Delivery : {prob_late*100:.1f}%")
print(f"  Model used     : {best_name}")